# DataMate — QLoRA trên Kaggle T4 ×2, rồi xuất public_test

Chạy từ đầu với **GPU T4 x2**, Internet **On** khi cài thư viện/tải model, Persistence **Files only**.
Input: bộ chính thức có `train/`, `public_test/` và checkpoint Qwen2.5-Coder-7B-Instruct-AWQ.

1. Cài phiên bản thư viện đã ghi nhận trong log; kiểm tra TrainingArguments trước khi chạy GPU.
2. Smoke test suy luận, rồi chạy 3.000 câu train trên hai GPU.
3. Tạo SFT từ SQL được thực thi lại và khớp đáp án; giữ riêng 6 business.
4. Tải base FP16, smoke test DDP 2 bước rồi train 2 epoch; loss chỉ trên SQL.
5. So base và adapter từng epoch trên **toàn bộ câu của 6 business giữ lại**; chọn theo đúng − sai.
6. Suy luận public_test offline trên hai GPU; kiểm tra đủ ID rồi xuất submission và log.

Mốc lần trước: baseline 175 phút, 1.156/3.000 đúng, điểm −218, 888 mẫu SFT trước lọc token.
Đây không phải cam kết thời gian hay điểm sau fine-tune. Public_test không dùng để train/chọn checkpoint.
Source sửa phải được push lên `BRANCH` trước khi Run All. Xem `docs/19_finetune_plan.md`.

Bản sửa OOM: logits/loss chỉ cho SQL, vẫn giữ toàn bộ context. Smoke test dùng các mẫu dài nhất và báo peak VRAM từng rank. Có thể dùng STAGE1_ARCHIVE để phục hồi baseline cũ. Public_test không có gold; nộp submission để nhận điểm chính thức.

Sau baseline, tạo submission_base.jsonl trước khi train để bạn có file nộp xem điểm sớm. Khi train và chọn model xong, submission.jsonl là bản được chọn. Không suy ra điểm chính thức từ tỷ lệ câu trả lời.


In [ ]:
# Cell 1 — lấy source code MỚI NHẤT (clone lần đầu; nếu thư mục đã có thì fetch + reset về origin/BRANCH)
import os, subprocess, pathlib
REPO_URL = "https://github.com/HoL0ngg/Zalo-AI-Challenge-2026.git"
BRANCH = "main"
ROOT = pathlib.Path("/kaggle/working/datamate")

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=True).stdout.strip()

if (ROOT / ".git").exists():
    git("remote", "set-url", "origin", REPO_URL, cwd=ROOT)
    git("fetch", "--depth", "1", "origin", BRANCH, cwd=ROOT)
    git("reset", "--hard", "FETCH_HEAD", cwd=ROOT)      # bỏ thay đổi cục bộ, lấy đúng code trên GitHub
    git("clean", "-fd", "-e", "models/", cwd=ROOT)       # giữ symlink trọng số ở models/
else:
    if ROOT.exists():
        import shutil; shutil.rmtree(ROOT)               # thư mục dở dang, không phải repo
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT))
os.chdir(ROOT)
remote = git("ls-remote", REPO_URL, BRANCH).split()[0]   # commit thật sự đang ở đầu nhánh trên GitHub
local = git("rev-parse", "HEAD", cwd=ROOT)
print("local :", git("log", "--oneline", "-1", cwd=ROOT))
print("remote:", remote[:7], "(đầu nhánh", BRANCH, "trên GitHub)")
assert local == remote, "Code chưa mới nhất: kiểm tra đã git push chưa và BRANCH đúng chưa"
# P0 (sql_parser) và data parallel (dataparallel) phải có, nếu không là đang chạy code cũ.
for needed in ("src/datamate/codegen/sql_parser.py", "src/datamate/runtime/dataparallel.py"):
    assert (ROOT / needed).exists(), f"Thiếu {needed}: chưa push code lên GitHub"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)


In [ ]:

from pathlib import Path

# Chỉ đọc dataset chính thức, KHÔNG quét model Qwen hay datamate
DATASET_ROOT = Path(
    "/kaggle/input/datasets/longgh/zalo-dataset/Zalo_dataset"
)

TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "public_test"

# Kiểm tra cấu trúc dữ liệu
for name, d in [("train", TRAIN_DIR), ("public_test", TEST_DIR)]:
    assert d.is_dir(), f"Không tìm thấy folder: {d}"

    for required in ["questions.jsonl", "rulebook.md", "calendar.csv"]:
        assert (d / required).exists(), f"Thiếu {required} trong {name}"

    assert (d / "data").is_dir(), f"Thiếu data/ trong {name}"

    num_questions = sum(
        1 for _ in open(d / "questions.jsonl", encoding="utf-8")
    )
    print(f"{name}: {num_questions} questions")
    print(f"  Path: {d}")

assert (TRAIN_DIR / "answers.jsonl").exists(), \
    "Train phải có answers.jsonl"

assert not (TEST_DIR / "answers.jsonl").exists(), \
    "Public test không được có answers.jsonl"

In [ ]:
# Các siêu tham số cho T4 16 GB ×2; giữ cố định trước khi đo holdout.
from types import SimpleNamespace
TRAIN = SimpleNamespace(out=pathlib.Path('/kaggle/working/lora'), max_len=4096,
                        epochs=2.0, lr=5e-5, rank=32, accum=8, seed=20261030, max_steps=-1)
# Batch hiệu dụng = 1 sequence/GPU × 2 GPU × accum 8 = 16.
# shortcut: max_len 4096 loại mẫu dài; chỉ tăng sau khi đo VRAM T4.
# Không cắt mất prompt/SQL để vừa ngưỡng; script báo số mẫu bị loại.

# Optional: Add Input chứa stage1_sft.zip của lần chạy 53dafa3, rồi điền đường dẫn.
# Giữ None để chạy baseline mới; không dùng archive nếu đã đổi pipeline inference/rulebook/data.
STAGE1_ARCHIVE = None


In [ ]:
# Cell 2a — tìm chính xác thư mục AWQ trong Kaggle Input (không copy cả Dataset)
from pathlib import Path
import json
matches = []
for config in Path('/kaggle/input').rglob('config.json'):
    folder = config.parent
    try:
        meta = json.loads(config.read_text())
    except (ValueError, OSError):
        continue
    quant = meta.get('quantization_config') or {}
    if (quant.get('quant_method', '').lower() == 'awq' and
        any(folder.glob('*.safetensors'))):
        matches.append(folder)
print('AWQ checkpoints:', *map(str, matches), sep='\n  ')
assert matches, 'Không tìm thấy checkpoint AWQ; kiểm tra Add Input model Qwen'
assert len(matches) == 1, f'Tìm thấy nhiều checkpoint AWQ, hãy chọn thủ công: {matches}'
MODEL_AWQ = matches[0]


In [ ]:
# Cell 2b — cấu hình trỏ trực tiếp vào Input (read-only) qua symlink, KHÔNG copy 5+ GB
from pathlib import Path
import os
model_link = ROOT / 'models' / 'qwen-coder-awq'
model_link.parent.mkdir(parents=True, exist_ok=True)
if model_link.is_symlink() or model_link.is_file():
    model_link.unlink()
elif model_link.exists():
    raise RuntimeError(f'{model_link} là thư mục thật: không tự xóa, hãy kiểm tra trước')
model_link.symlink_to(MODEL_AWQ, target_is_directory=True)
assert (model_link / 'config.json').exists(), 'Model symlink bị hỏng'
assert any(model_link.glob('*.safetensors')), 'Thiếu safetensors'
print('AWQ model:', model_link, '->', model_link.resolve())


In [ ]:
# Cài thư viện trước khi import torch; kiểm tra API train ngay tại đây.
import sys, subprocess, importlib
from importlib.metadata import version
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]',
                '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchaudio'], check=True)
import torch

def torchvision_ok():
    try:
        for name in list(sys.modules):
            if name == 'torchvision' or name.startswith('torchvision.'):
                del sys.modules[name]
        importlib.import_module('torchvision.ops')
        from torchvision.transforms import InterpolationMode
        return True
    except Exception as exc:
        print('torchvision:', repr(exc)[:200])
        return False

if not torchvision_ok():
    minor = int(torch.__version__.split('.')[1])
    tag = 'cu' + torch.version.cuda.replace('.', '')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchvision'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    f'torchvision==0.{minor + 15}.*', '--index-url',
                    f'https://download.pytorch.org/whl/{tag}'], check=True)
assert torchvision_ok(), 'torchvision không tương thích với torch; dừng trước baseline'
from scripts.train_lora import training_options
from transformers import TrainingArguments
import inspect
from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM
from transformers.loss.loss_utils import ForCausalLMLoss
assert "logits_to_keep" in inspect.signature(Qwen2ForCausalLM.forward).parameters
assert "shift_labels" in inspect.signature(ForCausalLMLoss).parameters
TrainingArguments(**training_options(TRAIN, 888, 2), use_cpu=True)
assert torch.cuda.device_count() == 2, 'Cần Accelerator GPU T4 x2'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])
packages = ['torch', 'torchvision', 'vllm', 'transformers', 'peft', 'bitsandbytes', 'accelerate']
print({name: version(name) for name in packages})
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True,
               stdout=open('/kaggle/working/environment.txt', 'w'))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


In [ ]:
# Cell 4 — preflight trước khi chạy: verify AWQ local_path trong cấu hình
import pathlib
assert (ROOT / 'models/qwen-coder-awq/config.json').exists(), 'models/qwen-coder-awq chưa có checkpoint AWQ'
assert pathlib.Path(TRAIN_DIR, 'questions.jsonl').exists(), 'Missing train questions'
print('Preflight: model AWQ + train input OK')


In [ ]:
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1', PYTHONHASHSEED='20261030')
# Cell 4b — smoke-test 1 business trước khi chạy 3.000 câu; in nguyên log nếu lỗi
import json, subprocess, sys
from pathlib import Path
sample = json.loads((TRAIN_DIR / 'questions.jsonl').read_text(encoding='utf-8').splitlines()[0])
business = sample['business_id']
SMOKE_DIR = Path('/kaggle/working/result/smoke')
smoke_cmd = [sys.executable, '-m', 'datamate.runtime.dataparallel',
             '--config', 'configs/kaggle.yaml', '--data-dir', str(TRAIN_DIR),
             '--result-dir', str(SMOKE_DIR), '--gpus', '0',
             '--business', business, '--max-per-business', '1']
print('Smoke-test business:', business, flush=True)
proc = subprocess.run(smoke_cmd, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout[-5000:])
for logfile in sorted((SMOKE_DIR / '.shards').glob('shard_*.log')):
    txt = logfile.read_text(errors='replace')
    print('\n===', logfile.name, '===\n', txt[-12000:])
assert proc.returncode == 0, 'Smoke test lỗi: xem traceback vừa được in ở trên'
assert (SMOKE_DIR / 'submission.jsonl').exists(), 'Smoke test không tạo submission'
print('Smoke test passed')


In [ ]:
# Cell 5 — STAGE 1: subprocess có kiểm tra exit code và in shard tracebacks
import os, sys, time, subprocess, pathlib
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1', PYTHONHASHSEED='20261030')
R1 = '/kaggle/working/result/stage1'
if STAGE1_ARCHIVE is not None:
    from scripts.build_sft_data import restore_baseline
    restore_baseline(pathlib.Path(STAGE1_ARCHIVE), TRAIN_DIR, pathlib.Path(R1))
    print('Đã phục hồi baseline; SFT sẽ được tạo và kiểm tra lại')
else:
    _t0 = time.time()
    command = [sys.executable, '-m', 'datamate.runtime.dataparallel',
               '--config', 'configs/kaggle.yaml', '--data-dir', str(TRAIN_DIR),
               '--result-dir', R1, '--gpus', '0,1']
    run = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('Stage 1 summary (last 4000 chars):\n', run.stdout[-4000:])
    print(f'Stage 1 elapsed: {(time.time() - _t0)/60:.1f} min')
    logfiles = sorted(pathlib.Path(R1, '.shards').glob('shard_*.log'))
    failed = []
    for log in logfiles:
        content = log.read_text(errors='replace')
        if 'Traceback' in content or 'ERROR' in content or run.returncode:
            failed.append(log.name)
            print(f'\n===== {log.name} (tail) =====\n{content[-14000:]}')
    if run.returncode:
        raise RuntimeError(f'Data-parallel exit code {run.returncode}. Failed shards: {failed}. See logs above.')
    if not pathlib.Path(R1, 'submission.jsonl').exists():
        raise RuntimeError('Stage 1 thiếu submission.jsonl, kiểm tra log shards ở trên')

def checked_cli(*args, output_file=None):
    cp = subprocess.run([sys.executable, '-m', 'datamate.cli', *map(str,args)], cwd=ROOT,
                        text=True, capture_output=True)
    if cp.returncode:
        raise RuntimeError(f'CLI error: {cp.stderr[-8000:]}')
    if output_file:
        pathlib.Path(output_file).write_text(cp.stdout, encoding='utf-8')
    else:
        print(cp.stdout[-2000:])
checked_cli('eval', f'{R1}/submission.jsonl', TRAIN_DIR / 'answers.jsonl', output_file='/kaggle/working/eval_stage1.json')
checked_cli('report', R1, TRAIN_DIR / 'answers.jsonl', output_file='/kaggle/working/report_stage1.json')
print(pathlib.Path('/kaggle/working/report_stage1.json').read_text()[:1800])

# Tạo file nộp base trước khi train: nếu train lỗi, vẫn có kết quả public_test để nộp xem điểm.
import shutil
PUBLIC_BASE = '/kaggle/working/result/public_base'
cp = subprocess.run([sys.executable, '-m', 'datamate.runtime.dataparallel', '--config', 'configs/kaggle.yaml',
                     '--data-dir', str(TEST_DIR), '--result-dir', PUBLIC_BASE, '--gpus', '0,1'],
                    cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(cp.stdout[-4000:])
if cp.returncode:
    for p in sorted(pathlib.Path(PUBLIC_BASE, '.shards').glob('shard_*.log')):
        print(p.name, p.read_text(errors='replace')[-9000:])
    raise RuntimeError('Base public_test failed; xem shard logs')
checked_cli('validate', f'{PUBLIC_BASE}/submission.jsonl')
public_ids = {json.loads(line)['id'] for line in (TEST_DIR / 'questions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()}
base_public = [json.loads(line) for line in pathlib.Path(PUBLIC_BASE, 'submission.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(base_public) == len(public_ids) and {r['id'] for r in base_public} == public_ids, 'Base public_test thiếu/trùng ID'
shutil.copy2(f'{PUBLIC_BASE}/submission.jsonl', '/kaggle/working/submission_base.jsonl')
shutil.copy2(f'{PUBLIC_BASE}/submission.jsonl', '/kaggle/working/submission.jsonl')
print('Đã có submission_base.jsonl để nộp xem điểm; public_test không có gold để chấm tại notebook')


In [ ]:
# SFT: chỉ train có đáp án; script thực thi lại SQL trước khi giữ mẫu.
import json, subprocess, sys, zipfile
subprocess.run([sys.executable, 'scripts/build_sft_data.py', R1, str(TRAIN_DIR),
                '--out', '/kaggle/working/sft/train.jsonl',
                '--holdout-out', '/kaggle/working/sft/holdout.json'], check=True)
HOLDOUT = json.loads(pathlib.Path('/kaggle/working/sft/holdout.json').read_text())['businesses']
N_SFT = sum(1 for _ in open('/kaggle/working/sft/train.jsonl', encoding='utf-8'))
print('SFT:', N_SFT, '| holdout:', HOLDOUT)
assert len(HOLDOUT) == 6 and N_SFT >= 300, 'Kiểm tra baseline/SFT trước khi train'
# Bảo toàn log và SFT để có thể tái sử dụng nếu GPU train gặp lỗi.
with zipfile.ZipFile('/kaggle/working/stage1_sft.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (pathlib.Path(R1), pathlib.Path('/kaggle/working/sft')):
        for file in sorted(folder.rglob('*')):
            if file.is_file() and '.shards' not in file.parts:
                z.write(file, file.relative_to('/kaggle/working'))


In [ ]:
# Base chưa lượng tử để train; inference vẫn dùng AWQ. Chỉ download ở cell chuẩn bị này.
import os, shutil, pathlib, json
BASE_FP16 = pathlib.Path('/tmp/qwen-fp16')
hits = []
for c in pathlib.Path('/kaggle/input').rglob('config.json'):
    meta = json.loads(c.read_text())
    if not meta.get('quantization_config') and any(c.parent.glob('*.safetensors')) and 'coder' in str(c.parent).lower():
        hits.append(c.parent)
if hits:
    assert len(hits) == 1, f'Nhiều base FP16, hãy chọn rõ: {hits}'
    BASE_FP16 = hits[0]
else:
    assert shutil.disk_usage('/tmp').free / 1e9 > 25, 'Cần >25 GB trống ở /tmp'
    # Stage 1 đã bật offline; chỉ tạm tắt cho download có chủ ý, trong process riêng.
    download = "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen2.5-Coder-7B-Instruct', local_dir='/tmp/qwen-fp16', allow_patterns=['*.json','*.safetensors','*.txt'])"
    subprocess.run([sys.executable, '-c', download], check=True,
                   env=dict(os.environ, HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0'))
assert (BASE_FP16 / 'tokenizer.json').is_file(), 'Thiếu tokenizer base'
print('Base FP16:', BASE_FP16)


In [ ]:
# Smoke test DDP 2 bước, sau đó train đầy đủ; mỗi lệnh có log và exit code.
import os, subprocess, pathlib, time
from pathlib import Path
assert Path('/kaggle/working/sft/train.jsonl').is_file(), 'SFT missing'

def train_command(out, max_steps):
    return [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_lora.py', '--base', str(BASE_FP16),
            '--data', '/kaggle/working/sft/train.jsonl', '--out', str(out),
            '--max-len', str(TRAIN.max_len), '--epochs', str(TRAIN.epochs), '--lr', str(TRAIN.lr),
            '--rank', str(TRAIN.rank), '--accum', str(TRAIN.accum), '--seed', str(TRAIN.seed),
            '--max-steps', str(max_steps)]

for name, out, steps in [('smoke', Path('/tmp/datamate-lora-smoke'), 2),
                         ('train', TRAIN.out, -1)]:
    log = Path(f'/kaggle/working/qlora_{name}.log')
    t0 = time.monotonic()
    with log.open('w', encoding='utf-8') as f:
        result = subprocess.run(train_command(out, steps), cwd=ROOT, stdout=f, stderr=subprocess.STDOUT,
                                env=dict(os.environ, TOKENIZERS_PARALLELISM='false', PYTORCH_ALLOC_CONF='expandable_segments:True'))
    print(name, 'exit:', result.returncode, 'minutes:', round((time.monotonic()-t0)/60, 1))
    print(log.read_text(encoding='utf-8', errors='replace')[-12000:])
    if result.returncode:
        raise RuntimeError(f'DDP {name} failed; xem {log}. Baseline/SFT đã lưu trong stage1_sft.zip')
for filename in ('adapter_config.json', 'adapter_model.safetensors'):
    assert (TRAIN.out / filename).is_file(), f'Missing {filename}'
print('Train summary:', (TRAIN.out / 'training_summary.json').read_text())


In [ ]:
# Chọn checkpoint bằng điểm thực thi trên TOÀN BỘ 6 business holdout.
# Không dùng loss train hay kết quả public_test để chọn.
import json, subprocess, sys, yaml
from datamate.eval.scorer import score

def read_rows(p):
    return [json.loads(line) for line in pathlib.Path(p).read_text(encoding='utf-8').splitlines() if line.strip()]

gold = {r['id']: r for r in read_rows(TRAIN_DIR / 'answers.jsonl')}
ids = [q['id'] for q in read_rows(TRAIN_DIR / 'questions.jsonl') if q['business_id'] in HOLDOUT]
assert ids, 'Holdout rỗng'
expected = [gold[i] for i in ids]
base = {r['id']: r for r in read_rows(f'{R1}/submission.jsonl')}
assert all(i in base for i in ids), 'Baseline thiếu holdout, không so trên tập câu thiếu'
best = score([base[i] for i in ids], expected)
best['points'] = best['correct'] - best['wrong']
best['accuracy'] = best['correct'] / len(ids)
best['normalized_points'] = best['points'] / len(ids)
best_name, CONFIG = 'base', 'configs/kaggle.yaml'
selection = {'questions': len(ids), 'holdout_businesses': HOLDOUT, 'candidates': {'base': best}}
checkpoints = sorted((TRAIN.out / 'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
assert checkpoints, 'Không có checkpoint theo epoch'

for adapter in checkpoints:
    adapter_meta = json.loads((adapter / 'adapter_config.json').read_text())
    assert adapter_meta['r'] <= 32, 'Cập nhật khai báo ngân sách tham số nếu tăng rank'
    config = yaml.safe_load((ROOT / 'configs/models_kaggle_lora.yaml').read_text())
    config['models']['code_generation']['lora_path'] = str(adapter)
    config['models']['code_generation']['local_path'] = str((ROOT / config['models']['code_generation']['local_path']).resolve())
    models_file = Path('/kaggle/working') / f'models_{adapter.name}.yaml'
    models_file.write_text(yaml.safe_dump(config), encoding='utf-8')
    runtime = yaml.safe_load((ROOT / 'configs/kaggle_lora.yaml').read_text())
    runtime['models'] = str(models_file)
    config_file = Path('/kaggle/working') / f'{adapter.name}.yaml'
    config_file.write_text(yaml.safe_dump(runtime), encoding='utf-8')
    result_dir = Path('/kaggle/working/result') / f'holdout_{adapter.name}'
    cmd = [sys.executable, '-m', 'datamate.runtime.dataparallel', '--config', str(config_file),
           '--data-dir', str(TRAIN_DIR), '--result-dir', str(result_dir), '--gpus', '0,1', '--business', *HOLDOUT]
    cp = subprocess.run(cmd, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(cp.stdout[-4000:])
    if cp.returncode:
        for p in sorted((result_dir / '.shards').glob('shard_*.log')):
            print(p.name, p.read_text(errors='replace')[-9000:])
        raise RuntimeError(f'Holdout {adapter.name} failed; không chọn base để che lỗi chạy')
    tuned_rows = read_rows(result_dir / 'submission.jsonl')
    assert len(tuned_rows) == len(ids) and {r['id'] for r in tuned_rows} == set(ids), 'Holdout thiếu/trùng ID'
    candidate = score(tuned_rows, expected)
    candidate['points'] = candidate['correct'] - candidate['wrong']
    candidate['accuracy'] = candidate['correct'] / len(ids)
    candidate['normalized_points'] = candidate['points'] / len(ids)
    selection['candidates'][adapter.name] = candidate
    checked_cli('report', result_dir, TRAIN_DIR / 'answers.jsonl',
                output_file=f'/kaggle/working/report_{adapter.name}.json')
    print(adapter.name, candidate)
    if candidate['points'] > best['points']:
        best, best_name, CONFIG = candidate, adapter.name, str(config_file)

selection.update(selected=best_name, config=CONFIG)
Path('/kaggle/working/selection.json').write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')
print('Chọn:', best_name, '| đúng/sai/điểm:', best['correct'], best['wrong'], best['points'])
# Giữ adapter được chọn ở models/lora để dùng lại cấu hình repo sau phiên này.
if best_name != 'base':
    selected = TRAIN.out / 'checkpoints' / best_name
    for filename in ('adapter_config.json', 'adapter_model.safetensors'):
        shutil.copy2(selected / filename, TRAIN.out / filename)
    link = ROOT / 'models/lora'
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f'Không ghi đè thư mục thật {link}')
    link.symlink_to(TRAIN.out, target_is_directory=True)


In [ ]:
# Public_test: chỉ suy luận offline, hai GPU; dừng nếu tiến trình hoặc định dạng lỗi.
import os, time, shutil, json, pathlib
from collections import Counter
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1', PYTHONHASHSEED='20261030')
if CONFIG == 'configs/kaggle.yaml':
    RT = PUBLIC_BASE
    t0 = time.monotonic()
    print('Base thắng holdout: dùng lại public_base đã chạy')
else:
    RT = '/kaggle/working/result/public_test'
    t0 = time.monotonic()
    cp = subprocess.run([sys.executable, '-m', 'datamate.runtime.dataparallel', '--config', CONFIG,
                         '--data-dir', str(TEST_DIR), '--result-dir', RT, '--gpus', '0,1'],
                        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(cp.stdout[-4000:])
    if cp.returncode:
        for p in sorted(pathlib.Path(RT, '.shards').glob('shard_*.log')):
            print(p.name, p.read_text(errors='replace')[-9000:])
        raise RuntimeError('Public_test failed; kiểm tra shard logs')
checked_cli('validate', f'{RT}/submission.jsonl')
ids = {r['id'] for r in read_rows(TEST_DIR / 'questions.jsonl')}
sub = read_rows(f'{RT}/submission.jsonl')
assert len(sub) == len(ids) and {r['id'] for r in sub} == ids, 'Submission thiếu/trùng/thừa ID'
shutil.copy2(f'{RT}/submission.jsonl', '/kaggle/working/submission.jsonl')
print('Public_test:', len(sub), 'câu;', dict(Counter(r['type'] for r in sub)),
      '| phút:', round((time.monotonic() - t0) / 60, 1))


In [ ]:
# Đóng gói log, adapter, cấu hình và SFT để đọc kết quả/tái sử dụng.
import zipfile
working = Path('/kaggle/working')
files = []
for pattern in ('report_*.json', 'eval_*.json', 'selection.json', 'environment.txt', 'qlora_*.log',
                'models_checkpoint-*.yaml', 'checkpoint-*.yaml', 'submission.jsonl', 'submission_base.jsonl'):
    files.extend(working.glob(pattern))
for folder in ('lora', 'sft', 'result'):
    files.extend(p for p in (working / folder).rglob('*') if p.is_file() and '.shards' not in p.parts)
with zipfile.ZipFile(working / 'datamate_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for file in sorted(set(files)):
        z.write(file, file.relative_to(working))
print('Output:', working / 'datamate_results.zip')
